# Web server log: NASA Kennedy Space Center, tháng 7/1995

Log là nguồn **append-only, bán cấu trúc, khối lượng lớn**. Bộ này là access log thật của web
server NASA KSC: khoảng 1,9 triệu request trong ~20 MB gzip, định dạng Common Log Format:

```text
199.72.81.55 - - [01/Jul/1995:00:00:01 -0400] "GET /history/apollo/ HTTP/1.0" 200 6245
host         ident user  [timestamp kèm múi giờ]     "request line"            status bytes
```

Việc của bạn, theo đúng thứ tự một pipeline thật:

1. **parse**: đọc streaming từng dòng (không `readlines()`), dòng không khớp thì tách ra danh sách reject, không làm sập pipeline;
2. **clean**: ép kiểu, đưa timestamp về UTC, phân loại host;
3. **insights**: trả lời các câu hỏi vận hành bằng số;
4. **write_clean**: ghi Parquet vào `data/clean/nasa-http-logs/`, chạy lại không nhân đôi dòng;
5. vẽ biểu đồ, rồi viết `report.md` cạnh notebook.

Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ
qua, test gọi thẳng các hàm bạn viết. Chấm:
`uv run pytest tracks/data-engineer/modules/00-introduction/exercises/basic/02-web-server-logs -q`

In [ ]:
# explore: tìm gốc repo (thư mục có pyproject.toml), rồi tải bộ dữ liệu nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

RAW = ensure("nasa-http-logs")
CLEAN = ROOT / "data" / "clean" / "nasa-http-logs"
print(RAW, sorted(p.name for p in RAW.iterdir()))

In [ ]:
import gzip
import re
from collections.abc import Iterable
from pathlib import Path

import numpy as np
import pandas as pd

COLUMNS = ["host", "ts", "method", "path", "protocol", "status", "bytes"]
LOCAL_TZ = "America/New_York"   # -0400 trong tháng 7 (EDT), giờ của server ở Florida

## 1. Parse từng dòng

`parse_line(line)` trả `dict` với đúng các khoá trong `COLUMNS`, hoặc `None` nếu dòng hỏng:

- `ts` giữ **nguyên chuỗi** `"01/Jul/1995:00:00:01 -0400"` (đổi kiểu hàng loạt ở bước clean nhanh hơn nhiều lần so với `strptime` từng dòng);
- `status` là `int`; `bytes` là `int`, và `-` (response không có body) thành `0`;
- request line tách theo khoảng trắng: token đầu là `method`; token cuối là `protocol` nếu bắt đầu bằng `HTTP/`,
  không thì `protocol = None` (request kiểu HTTP/0.9: `"GET /page.html"`); phần giữa là `path`;
- request rỗng hoặc chỉ có một token là dòng hỏng.

Log thật có request chứa dấu `"` bên trong (`"GET /images/" HTTP/1.0"`). Regex của bạn nên vẫn đọc được dòng đó.

In [ ]:
LOG_RE = re.compile(r'^(\S+) \S+ \S+ \[([^\]]+)\] "(.*)" (\d{3}) (\d+|-)\s*$')


def parse_line(line: str) -> dict | None:
    """Một dòng Common Log Format -> dict theo COLUMNS, hoặc None nếu dòng hỏng."""
    match = LOG_RE.match(line)
    if match is None:
        return None
    host, ts, request, status, size = match.groups()
    parts = request.split()
    if len(parts) < 2:
        return None
    protocol = parts[-1] if len(parts) >= 3 and parts[-1].startswith("HTTP/") else None
    path = " ".join(parts[1:-1] if protocol else parts[1:])
    return {"host": host, "ts": ts, "method": parts[0], "path": path, "protocol": protocol,
            "status": int(status), "bytes": 0 if size == "-" else int(size)}


def parse_log(lines: Iterable[str]) -> tuple[pd.DataFrame, list[str]]:
    """Đọc streaming; trả (DataFrame cột COLUMNS, danh sách dòng hỏng đã bỏ ký tự xuống dòng)."""
    rows, rejected = [], []
    for line in lines:
        row = parse_line(line)
        if row is None:
            rejected.append(line.rstrip("\n"))
        else:
            rows.append(row)
    return pd.DataFrame.from_records(rows, columns=COLUMNS), rejected

In [ ]:
# explore: đọc thẳng file gzip theo dòng, không giải nén ra đĩa
with gzip.open(RAW / "NASA_access_log_Jul95.gz", "rt", encoding="latin-1") as fh:
    raw_df, rejected = parse_log(fh)
print(f"{len(raw_df):,} request, {len(rejected)} dòng hỏng")
for line in rejected[:10]:
    print(repr(line))

## 2. Làm sạch

`clean(df)` trả bản mới (không sửa `df` gốc):

- `ts`: `datetime64` **UTC** (đọc chuỗi có offset `-0400` rồi `tz_convert("UTC")`). Lưu UTC, chỉ đổi về giờ địa phương khi phân tích;
- `status`, `bytes`: kiểu số nguyên;
- thêm cột `host_type`: `"ip"` nếu host là địa chỉ IPv4, `"local"` nếu host không có dấu chấm,
  còn lại là đuôi tên miền viết thường (`"com"`, `"edu"`, `"gov"`, `"jp"`...);
- sắp theo `ts`. **Không** xoá dòng trùng: log không có khoá, hai request giống hệt trong cùng giây là hai request thật.

In [ ]:
IPV4 = re.compile(r"\d{1,3}(?:\.\d{1,3}){3}")


def host_type(host: str) -> str:
    if IPV4.fullmatch(host):
        return "ip"
    if "." not in host:
        return "local"
    return host.rsplit(".", 1)[1].lower()


def clean(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["ts"] = pd.to_datetime(out["ts"], format="%d/%b/%Y:%H:%M:%S %z").dt.tz_convert("UTC")
    out["status"] = out["status"].astype("int16")
    out["bytes"] = out["bytes"].astype("int64")
    types = {h: host_type(h) for h in out["host"].unique()}   # ~80 nghìn host, nhanh hơn map từng dòng
    out["host_type"] = out["host"].map(types)
    return out.sort_values("ts", kind="stable").reset_index(drop=True)

In [ ]:
# explore: dữ liệu sau khi làm sạch
df = clean(raw_df)
df.info()
df.head()

## 3. Insight

`insights(df, rejected)` trả `dict` với đúng các khoá dưới. Ngày và giờ tính theo **giờ địa phương
của server** (`LOCAL_TZ`); chuỗi thời gian dạng `"YYYY-MM-DD HH:MM:SS"`, ngày dạng `"YYYY-MM-DD"`.

| Khoá | Nghĩa |
| --- | --- |
| `requests`, `rejected`, `reject_rate` | số dòng đọc được, số dòng hỏng, tỉ lệ hỏng trên tổng số dòng |
| `unique_hosts`, `total_gb` | số host khác nhau, tổng byte / 1e9 |
| `not_found`, `top_404_path`, `top_404_count` | số request 404, đường dẫn 404 nhiều nhất và số lần |
| `peak_day`, `peak_day_requests` | ngày nhiều request nhất và số request ngày đó |
| `busiest_hour`, `quietest_hour` | giờ trong ngày (0–23) có tổng request lớn nhất / nhỏ nhất |
| `weekday_avg`, `weekend_avg` | số request trung bình mỗi ngày thường / mỗi ngày cuối tuần (tính trên các ngày có trong log) |
| `last_request` | thời điểm request cuối cùng |
| `max_gap_minutes`, `max_gap_start` | khoảng lặng dài nhất giữa hai request liên tiếp (phút) và thời điểm bắt đầu |
| `top_bytes_host_type` | `host_type` tải nhiều byte nhất |

In [ ]:
def insights(df: pd.DataFrame, rejected: list[str]) -> dict:
    local = df["ts"].dt.tz_convert(LOCAL_TZ)
    daily = df.groupby(local.dt.floor("D")).size()
    hourly = df.groupby(local.dt.hour).size()
    weekend = daily.index.day_of_week >= 5
    not_found = df.loc[df["status"] == 404, "path"].value_counts()

    ts = local.sort_values().reset_index(drop=True)
    gaps = ts.diff()
    k = gaps.idxmax()                     # vị trí request ngay SAU khoảng lặng
    fmt = "%Y-%m-%d %H:%M:%S"

    total = len(df) + len(rejected)
    return {
        "requests": len(df),
        "rejected": len(rejected),
        "reject_rate": len(rejected) / total if total else 0.0,
        "unique_hosts": int(df["host"].nunique()),
        "total_gb": float(df["bytes"].sum()) / 1e9,
        "not_found": int(not_found.sum()),
        "top_404_path": not_found.index[0],
        "top_404_count": int(not_found.iloc[0]),
        "peak_day": daily.idxmax().strftime("%Y-%m-%d"),
        "peak_day_requests": int(daily.max()),
        "busiest_hour": int(hourly.idxmax()),
        "quietest_hour": int(hourly.idxmin()),
        "weekday_avg": float(daily[~weekend].mean()),
        "weekend_avg": float(daily[weekend].mean()),
        "last_request": local.max().strftime(fmt),
        "max_gap_minutes": gaps[k].total_seconds() / 60,
        "max_gap_start": ts[k - 1].strftime(fmt),
        "top_bytes_host_type": df.groupby("host_type")["bytes"].sum().idxmax(),
    }

In [ ]:
# explore: các con số để viết report.md
INSIGHTS = insights(df, rejected)
INSIGHTS

## 4. Biểu đồ

Vẽ ít nhất hai biểu đồ (mỗi biểu đồ một cell `explore`):

1. số request **theo ngày** cả tháng: ngày nào vọt lên, vì sao? (tra lịch phóng tàu con thoi tháng 7/1995);
2. số request **theo giờ trong ngày**, tách ngày thường và cuối tuần: người dùng là ai, ở đâu?
3. (tuỳ chọn) top đường dẫn 404, hoặc byte theo `host_type`.

In [ ]:
# explore: request theo ngày, đánh dấu ngày phóng STS-70
import matplotlib.pyplot as plt

local = df["ts"].dt.tz_convert(LOCAL_TZ)
daily = df.groupby(local.dt.floor("D")).size()
fig, ax = plt.subplots(figsize=(10, 3.5))
colors = ["tab:orange" if d.day_of_week >= 5 else "tab:blue" for d in daily.index]
ax.bar(daily.index.strftime("%d"), daily.values, color=colors)
ax.annotate("13/07: phóng STS-70", xy=(12, daily.max()), xytext=(15, daily.max() * 0.95),
            arrowprops={"arrowstyle": "->"})
ax.set_title("Request mỗi ngày, NASA KSC 07/1995 (cam = cuối tuần; log dừng 28/07 13:32)")
ax.set_xlabel("ngày")
ax.set_ylabel("request")
plt.show()

In [ ]:
# explore: nhịp trong ngày, ngày thường vs cuối tuần (trung bình mỗi ngày)
is_weekend = local.dt.day_of_week >= 5
per_hour = df.groupby([is_weekend, local.dt.hour]).size().unstack(0)
n_days = daily.groupby(daily.index.day_of_week >= 5).size()
per_hour = per_hour / n_days.values
ax = per_hour.rename(columns={False: "ngày thường", True: "cuối tuần"}).plot(figsize=(10, 3.5), marker="o")
ax.set_title("Request trung bình theo giờ (giờ miền Đông Mỹ)")
ax.set_xlabel("giờ")
ax.set_ylabel("request / ngày")
plt.show()

In [ ]:
# explore: byte theo loại host (GB)
(df.groupby("host_type")["bytes"].sum().nlargest(8) / 1e9).sort_values().plot.barh(
    figsize=(8, 3.5), title="GB tải về theo loại host (top 8)")
plt.show()

## 5. Ghi dữ liệu sạch

`write_clean(df, out_dir)` ghi **một** tệp `out_dir/requests.parquet` (ghi đè, tạo thư mục nếu
chưa có) và trả đường dẫn. Ghi ra tệp tạm rồi `replace` để không bao giờ để lại tệp dở. Chạy notebook
hai lần thì `data/clean/nasa-http-logs/` vẫn đúng số dòng: pipeline **idempotent**.

In [ ]:
def write_clean(df: pd.DataFrame, out_dir: Path) -> Path:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / "requests.parquet"
    tmp = out_dir / ".requests.parquet.tmp"
    df.to_parquet(tmp, index=False)
    tmp.replace(path)
    return path

In [ ]:
# explore: ghi data/clean/nasa-http-logs/requests.parquet
path = write_clean(df, CLEAN)
print(path, f"{path.stat().st_size / 1e6:.1f} MB", len(pd.read_parquet(path)), "dòng")

## 6. Báo cáo

Viết `report.md` cạnh notebook này, đủ 4 mục `## Bộ dữ liệu`, `## Làm sạch`, `## Insight`,
`## Giới hạn`. Insight phải có số (lấy từ `INSIGHTS`), và mỗi con số kèm một câu "vì sao / nghĩa là gì".